# Compliance & Governance

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LakeLogic/LakeLogic/blob/main/examples/colab/02_compliance_governance.ipynb) [![View on GitHub](https://img.shields.io/badge/github-view_source-black?logo=github)](https://github.com/lakelogic/LakeLogic/blob/main/examples/colab/02_compliance_governance.ipynb)

GDPR erasure in 2 lines, automatic lineage, and per-entity cost intelligence.

In [ ]:
import subprocess
import sys
import importlib
import urllib.request
import os

if importlib.util.find_spec("lakelogic") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "lakelogic[polars]"])
if not os.path.exists("_setup.py"):
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/LakeLogic/LakeLogic/main/examples/colab/_setup.py", "_setup.py"
    )
import _setup as s
import lakelogic as ll

---
## 1. GDPR Erasure — `forget_subjects()` in 2 Lines

**The Problem:** A customer submits an Article 17 erasure request. Your legal team needs proof that all PII was removed across every table — with a timestamped audit trail.

**The Solution:** Mark fields as `pii: true` in the contract. Call `forget_subjects()`. Done.

In [ ]:
from lakelogic.core.gdpr import forget_subjects

contract_path = s.write_contract(
    """
version: 1.0.0
dataset: customers

model:
  fields:
    - name: customer_id
      type: string
      required: true
    - name: name
      type: string
      pii: true
    - name: email
      type: string
      pii: true
    - name: phone
      type: string
      pii: true
    - name: lifetime_value
      type: float
""",
    "02_compliance_governance_demo/customers.yaml",
)

# Generate a dataset with PII
source_df = ll.DataGenerator(contract_path).generate(rows=100)
proc = ll.DataProcessor(contract_path, engine="polars")
good, _ = proc.run(source_df)

import polars as pl

sample_id = good["customer_id"][0]
print(f"Before erasure ({sample_id}):")

display(
    good.filter(pl.col("customer_id") == sample_id).select(["customer_id", "name", "email", "phone", "lifetime_value"])
)

In [ ]:
# The Proof — erasure + audit trail
erased = forget_subjects(
    good,
    proc.contract,
    subject_column="customer_id",
    subject_ids=[sample_id],
    erasure_strategy="hash",
)

print(f"After erasure (subject: {sample_id}):")
audit_cols = [
    c
    for c in erased.columns
    if "customer_id" in c or "name" in c or "email" in c or "phone" in c or "lifetime_value" in c or "_lakelogic_" in c
]
display(erased.filter(pl.col("customer_id") == sample_id).select(audit_cols[:8]))

print("\nPII fields hashed. Audit columns added. Erasure in 2 lines of code.")

---
## 2. Automatic Lineage — Trace Any Row to Its Source

**The Problem:** An auditor asks: "Where did this Gold-layer number come from?" You spend 3 hours tracing through ETL jobs.

**The Solution:** LakeLogic stamps every row with `_lakelogic_run_id`, `_lakelogic_loaded_at`, and `_lakelogic_source_path` — automatically.

In [ ]:
# Run a pipeline and inspect lineage columns
lineage_contract = s.write_contract(
    """
version: 1.0.0
dataset: orders_lineage
info:
  title: silver_orders
  target_layer: silver

model:
  fields:
    - name: order_id
      type: integer
      required: true
    - name: amount
      type: float
    - name: status
      type: string

quality:
  row_rules:
    - name: positive
      sql: "amount > 0"

lineage:
  enabled: true
  upstream: [bronze.raw_orders]

""",
    "02_compliance_governance_demo/lineage.yaml",
)

source_df = ll.DataGenerator(lineage_contract).generate(rows=50)
proc = ll.DataProcessor(lineage_contract, engine="polars")
good, bad = proc.run(source_df, source_path="bronze/raw_orders/")

In [ ]:
# The Proof — lineage columns on every row
lineage_cols = [c for c in good.columns if "_lakelogic_" in c]
print(f"Lineage columns added automatically: {lineage_cols}")
print()
display(good.select(["order_id"] + lineage_cols).head(5))
print("\nEvery row traceable to its source. Every run has a unique ID.")

---
## 3. Pipeline Cost Intelligence

**The Problem:** Your cloud bill grows 40% in a quarter but nobody knows which domain or pipeline is responsible.

**The Solution:** LakeLogic estimates per-entity cost in every run report. Configure `metadata.cost` with a provider (`manual` or `databricks`) and LakeLogic calculates cost attribution using DBU rates, cluster scaling, and run duration.

In [ ]:
# Run two contracts with cost tracking enabled
small = s.write_contract(
    """
version: 1.0.0
dataset: small_entity
info:
  title: bronze_small_entity
  domain: marketing
  system: google_analytics

metadata:
  domain: marketing
  system: google_analytics
  data_layer: bronze
  cost:
    provider: "manual"
    currency: "USD"
    rates:
      dbu_per_hour: 0.22

model:
  fields:
    - name: id
      type: integer
      required: true
    - name: value
      type: string
""",
    "02_compliance_governance_demo/small.yaml",
)

large = s.write_contract(
    """
version: 1.0.0
dataset: large_entity
info:
  title: bronze_large_entity
  domain: finance
  system: shopify

metadata:
  domain: finance
  system: shopify
  data_layer: bronze
  cost:
    provider: "manual"
    currency: "USD"
    rates:
      dbu_per_hour: 0.55
    cluster:
      min_nodes: 2
      max_nodes: 8
      scaling_assumption: "avg"

model:
  fields:
    - name: id
      type: integer
      required: true
    - name: value
      type: string
""",
    "02_compliance_governance_demo/large.yaml",
)

p1 = ll.DataProcessor(small, engine="polars")
p1.run(ll.DataGenerator(small).generate(rows=100))
r1 = p1.last_report

p2 = ll.DataProcessor(large, engine="polars")
p2.run(ll.DataGenerator(large).generate(rows=5000))
r2 = p2.last_report

# ── Per-Entity Cost Report ──────────────────────────────────────
print("Per-Entity Cost Attribution")
print("=" * 60)
for label, r in [("marketing / small_entity", r1), ("finance   / large_entity", r2)]:
    counts = r.get("counts", {})
    cost = r.get("estimated_cost", 0) or 0
    currency = r.get("cost_currency", "USD") or "USD"
    confidence = r.get("cost_confidence", "none")
    duration = r.get("run_duration_seconds", 0)
    print(f"  {label}")
    print(f"    Rows     : {counts.get('source', '?'):>6}")
    print(f"    Duration : {duration:.3f}s")
    print(f"    Cost     : {currency} {cost:.6f}  (confidence: {confidence})")
    print()

print("In production, these cost estimates flow into the run log")
print("and feed domain-level budget dashboards.")
print("\nConfigure cost.provider in _system.yaml:")
print("  manual       → duration × DBU rate × nodes")
print("  databricks   → queries system.billing.usage for exact costs")

## What You Just Saw

- **GDPR erasure** — hash/nullify PII with an audit trail, in 2 lines
- **Automatic lineage** — run ID and timestamp on every row, no config needed
- **Cost intelligence** — per-entity, per-domain attribution in every run report

---
## Go Deeper — Explore by Capability

Each notebook below maps to a pillar of LakeLogic's [Technical Capabilities](https://lakelogic.github.io/LakeLogic/#technical-capabilities):

| # | Notebook | What You'll See |
|---|---|---|
| 🛡️ | **[Data Quality & Trust](01_data_quality_trust.ipynb)** | Reconciliation proofs, Pydantic validation, SQL-first rules, SLO monitoring |
| 📜 | **[Compliance & Governance](02_compliance_governance.ipynb)** | GDPR erasure in 2 lines, automatic lineage, cost intelligence |
| ⚡ | **[Engine & Scale](03_engine_scale.ipynb)** | Same contract on Polars & DuckDB, incremental processing, dry run |
| 🔧 | **[Developer Experience](04_developer_experience.ipynb)** | Structured diagnostics, DDL generation, surgical resets, multi-channel alerts |
| 🧬 | **[Data Generation & AI](05_data_generation_ai.ipynb)** | Synthetic data, referential integrity, edge case injection, contract inference |
| 🔌 | **[Integrations](06_integrations.ipynb)** | dbt adapter, dlt sources, contract-driven quality gates on arrival |

> **Each notebook is self-contained** — pick the capability that matters most to you and run it independently.